# E-Commerce Exploratory Data Analysis & Business Intelligence

**Project:** E-Commerce Sales, Customer & Profitability Analytics  
**Role:** Senior Data Analyst  
**Objective:** Uncover core revenue drivers, margin dynamics, customer cohort retention, return root causes, and geographic hotspots across multi-year transactions.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Visualization aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10

DATA_DIR = '../data/cleaned'
df_cust = pd.read_csv(os.path.join(DATA_DIR, 'customers_cleaned.csv'))
df_prod = pd.read_csv(os.path.join(DATA_DIR, 'products_cleaned.csv'))
df_ord = pd.read_csv(os.path.join(DATA_DIR, 'orders_cleaned.csv'))
df_pay = pd.read_csv(os.path.join(DATA_DIR, 'payments_cleaned.csv'))
df_ret = pd.read_csv(os.path.join(DATA_DIR, 'returns_cleaned.csv'))

print('Cleaned datasets loaded successfully.')

## 1. Core Enterprise KPIs
Executive-level financial and operational summary metrics.

In [2]:
delivered_orders = df_ord[df_ord['order_status'] == 'Delivered']
total_revenue = df_ord['sales_amount'].sum()
delivered_revenue = delivered_orders['sales_amount'].sum()
total_profit = df_ord['profit_amount'].sum()
delivered_profit = delivered_orders['profit_amount'].sum()
total_orders = len(df_ord)
total_customers = df_ord['customer_id'].nunique()
aov = total_revenue / total_orders
profit_margin = (total_profit / total_revenue) * 100
return_rate = (len(df_ord[df_ord['order_status'] == 'Returned']) / total_orders) * 100
cancel_rate = (len(df_ord[df_ord['order_status'] == 'Cancelled']) / total_orders) * 100

kpi_summary = pd.DataFrame([
    {'Metric': 'Gross Revenue (All Orders)', 'Value': f'INR {total_revenue:,.2f}'},
    {'Metric': 'Net Revenue (Delivered)', 'Value': f'INR {delivered_revenue:,.2f}'},
    {'Metric': 'Total Profit (All Orders)', 'Value': f'INR {total_profit:,.2f}'},
    {'Metric': 'Total Delivered Profit', 'Value': f'INR {delivered_profit:,.2f}'},
    {'Metric': 'Total Order Count', 'Value': f'{total_orders:,}'},
    {'Metric': 'Total Active Customers', 'Value': f'{total_customers:,}'},
    {'Metric': 'Average Order Value (AOV)', 'Value': f'INR {aov:,.2f}'},
    {'Metric': 'Overall Profit Margin', 'Value': f'{profit_margin:.2f}%'},
    {'Metric': 'Order Return Rate', 'Value': f'{return_rate:.2f}%'},
    {'Metric': 'Order Cancellation Rate', 'Value': f'{cancel_rate:.2f}%'},
])
display(kpi_summary)

## 2. Category Performance & Profitability Breakdown

In [3]:
ord_with_cat = df_ord.merge(df_prod[['product_id', 'category', 'subcategory']], on='product_id', how='left')
cat_summary = ord_with_cat.groupby('category').agg(
    Total_Revenue=('sales_amount', 'sum'),
    Total_Profit=('profit_amount', 'sum'),
    Orders=('order_id', 'count'),
    Avg_Discount=('discount_percentage', 'mean')
).reset_index()
cat_summary['Profit_Margin_%'] = (cat_summary['Total_Profit'] / cat_summary['Total_Revenue']) * 100
cat_summary = cat_summary.sort_values(by='Total_Revenue', ascending=False)
display(cat_summary)

## 3. Customer Repeat Purchase & Retention Analysis

In [4]:
cust_orders = df_ord.groupby('customer_id').size()
repeat_customers = (cust_orders > 1).sum()
repeat_rate = (repeat_customers / len(cust_orders)) * 100

print(f'Total Customers Who Purchased: {len(cust_orders):,}')
print(f'Repeat Customers (>1 order):   {repeat_customers:,}')
print(f'Repeat Purchase Rate:          {repeat_rate:.2f}%')

## 4. Payment Method Adoption & Return Root Causes

In [5]:
pay_share = df_pay.groupby('payment_method')['payment_amount'].agg(['count', 'sum']).reset_index()
pay_share.columns = ['Payment_Method', 'Transaction_Count', 'Total_Volume_INR']
pay_share['Share_%'] = (pay_share['Total_Volume_INR'] / pay_share['Total_Volume_INR'].sum()) * 100
display(pay_share.sort_values(by='Total_Volume_INR', ascending=False))

ret_reasons = df_ret.groupby('return_reason')['refund_amount'].agg(['count', 'sum']).reset_index()
ret_reasons.columns = ['Return_Reason', 'Return_Count', 'Refund_Amount_INR']
display(ret_reasons.sort_values(by='Return_Count', ascending=False))